# 📘 Khóa học: HR Analyst với SQL + PostgreSQL (7 buổi)

## 🎯 Tổng quan chương trình

- **Buổi 1:** 🚀 Giới thiệu dự án
- **Buổi 2:** 🗄️ Schema & Setup database
- **Buổi 3:** 👥 Employee Overview Analysis
- **Buổi 4:** 📊 Turnover Analysis
- **Buổi 5:** 💰 Salary Analysis
- **Buổi 6:** 📈 Engagement & Advanced Analysis
- **Buổi 7:** ✅ Quiz HR Analytics SQL

---

# 📊 Buổi 4: Turnover Analysis

## 🎯 Mục tiêu buổi này
✅ Sử dụng DATE functions (EXTRACT, AGE) để tính thời gian

✅ Dùng CASE WHEN để phân loại loại nghỉ việc (Voluntary vs Involuntary)

✅ Tính churn rate (tỷ lệ rời bỏ) theo thời gian

✅ Phân tích nguyên nhân rời bỏ

✅ Thực hành 5 bài tập đơn giản + 4 bài nâng cao

✅ Ôn tập 5 quiz từ cơ bản → nâng cao

---

## 🌏 Turnover Analysis là gì?

**Turnover Analysis** (Phân tích tỷ lệ rời bỏ) là phân tích số lượng nhân viên rời khỏi công ty. Đây là chỉ số **cực kỳ quan trọng** trong HR vì:

### 📊 Tại sao Turnover quan trọng?
- **Chi phí cao:** Thay thế 1 nhân viên = 50-200% lương thường niên (tuyển, đào tạo, mất năng suất)
- **Mất kiến thức:** Nhân viên có kinh nghiệm rời = mất IP, process, customer relationships
- **Ảnh hưởng tinh thần:** Turnover cao → morale thấp → more turnover (vòng lặp)
- **Uy tín công ty:** High turnover = khó tuyển người giỏi

### 📈 Turnover Rate bình thường:
- **Tech startups:** 15-25% (do thay đổi nhanh, cơ hội khác)
- **Corporate:** 10-15% (ổn định hơn)
- **Retail/F&B:** 25-50% (công việc seasonal, lương thấp)
- **Công ty lớn Việt Nam:** 20-30% (nhiều việc nhưng lương cạnh tranh)

### ❓ Các câu hỏi thường được hỏi
- Năm nay rời bỏ bao nhiêu nhân viên?
- Phòng nào có turnover cao nhất?
- Người nào rời (tình nguyện vs bị sa thải)?
- Lý do rời là gì? (tìm việc tốt hơn, lương thấp, quản lý tệ...)
- Nhân viên trẻ hay cũ rời nhiều hơn?

---

## 🔍 BƯỚC 1: DATE Functions - EXTRACT & AGE

**DATE Functions** dùng để làm việc với ngày tháng.
Trong phân tích Turnover ta cần:

- Khi nhân viên rời công ty → leaving_date

- Nhân viên làm bao lâu → hired_date → leaving_date

### 1.1 EXTRACT - Lấy năm/tháng từ ngày
```sql
SELECT 
  emp_id,
  leaving_date,
  EXTRACT(YEAR FROM leaving_date) AS leave_year,
  EXTRACT(MONTH FROM leaving_date) AS leave_month
FROM leavers
LIMIT 5;
```
**Kỳ vọng:** Tách năm/tháng rời bỏ từ ngày.

### 1.2 AGE - Tính khoảng thời gian
```sql
SELECT 
  l.emp_id,
  e.hired_date,
  l.leaving_date,
  AGE(l.leaving_date, e.hired_date) AS tenure
FROM leavers l
JOIN employees e 
ON l.emp_id = e.emp_id
LIMIT 5;
```
**Kỳ vọng:** Tính số năm/tháng giữa 2 ngày (tenure = thời gian làm việc).

### 1.3 Kết hợp AGE + EXTRACT
```sql
SELECT 
  l.emp_id,
  EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) AS years_tenure,
  EXTRACT(MONTH FROM AGE(l.leaving_date, e.hired_date)) AS months_tenure
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
LIMIT 5;
```
**Kỳ vọng:** Tách năm/tháng từ tenure (có thể được nói: "A rời sau 3 năm 5 tháng").

**Giải thích:**
- `EXTRACT(YEAR FROM ...)` lấy năm từ ngày hoặc interval
- `AGE(date1, date2)` tính interval giữa 2 ngày (date1 - date2)
- Interval có thể extract tháng, ngày, v.v.

---

## 🔍 BƯỚC 2: CASE WHEN - Phân loại rời bỏ

**CASE WHEN** dùng để phân loại dữ liệu.

Trong HR Analytics thường phân loại:

- Voluntary → nhân viên tự nghỉ

- Involuntary → công ty cho nghỉ

Trong dataset này đã có sẵn cột exit_type.

### 2.1 Cơ bản: Tình nguyện vs Sa thải
```sql
SELECT 
  emp_id,
  leaving_date,
  reason,
  exit_type,
  CASE
    WHEN exit_type = 'Voluntary' THEN 'Employee Decision'
    WHEN exit_type = 'Involuntary' THEN 'Company Decision'
    WHEN exit_type = 'Retirement' THEN 'Retirement'
    ELSE 'Other'
  END AS separation_category
FROM leavers
LIMIT 10;
```
**Kỳ vọng:** Mỗi dòng ghi Voluntary/Involuntary/Unknown.

### 2.2 Phân loại mức độ (High Risk vs Low Risk)
```sql
SELECT 
  l.emp_id,
  EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) AS years_tenure,
  CASE
    WHEN EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) < 1 THEN 'High Risk (<1 year)'
    WHEN EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) < 3 THEN 'Medium Risk (1-3 years)'
    ELSE 'Low Risk (>3 years)'
  END AS risk_category
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
LIMIT 10;
```
**Kỳ vọng:** Nhân viên rời sớm (< 1 năm) = High Risk.

**Giải thích:**
- `IN ('val1', 'val2', ...)` kiểm tra nếu giá trị thuộc danh sách
- `WHEN ... THEN ...` là điều kiện và giá trị trả về
- `ELSE` là trường hợp mặc định nếu không match

---

## 🔍 BƯỚC 3: Churn Rate Calculation

**Churn Rate** = (Số nhân viên rời / Tổng nhân viên) × 100 %

### 3.1 Churn rate toàn công ty
```sql
SELECT 
  COUNT(DISTINCT l.emp_id) AS total_leavers,
  COUNT(DISTINCT e.emp_id) AS total_employees,
  ROUND(
    (COUNT(DISTINCT l.emp_id)::NUMERIC /
    COUNT(DISTINCT e.emp_id)) * 100,
    2
  ) AS churn_rate_pct
FROM leavers l
CROSS JOIN employees e;
```
**Kỳ vọng:** Ví dụ 100 rời / 1000 nhân viên = 10.00% churn rate.

### 3.2 Churn rate theo năm
```sql
SELECT 
  EXTRACT(YEAR FROM l.leaving_date) AS leave_year,
  COUNT(DISTINCT l.emp_id) AS leavers,
  ROUND(
    (COUNT(DISTINCT l.emp_id)::NUMERIC /
    COUNT(DISTINCT e.emp_id)) * 100,
    2
  ) AS churn_rate_pct
FROM leavers l
CROSS JOIN employees e
GROUP BY EXTRACT(YEAR FROM l.leaving_date)
ORDER BY leave_year;
```
**Kỳ vọng:** 

### 3.3 Churn rate theo phòng ban
```sql
SELECT 
  d.dept_name,
  COUNT(DISTINCT l.emp_id) AS leavers,
  COUNT(DISTINCT e.emp_id) AS total_emps,
  ROUND(
    COUNT(DISTINCT l.emp_id)::NUMERIC 
    / COUNT(DISTINCT e.emp_id) * 100,
    2
  ) AS churn_rate_pct
FROM employees e
LEFT JOIN leavers l 
  ON e.emp_id = l.emp_id
JOIN departments d 
  ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY churn_rate_pct DESC;
```
**Kỳ vọng:** 

**Giải thích:**
- `::NUMERIC` ép kiểu số thập phân để chia chính xác
- `CROSS JOIN` để lấy tổng employees
- `GROUP BY EXTRACT(YEAR FROM ...)` để nhóm theo năm

---

## 🔍 BƯỚC 4: Phân tích chi tiết Voluntary vs Involuntary

### 4.1 Tỉ lệ nghỉ tình nguyện với nghỉ sa thải
```sql
SELECT 
  exit_type,
  COUNT(*) AS count,
  ROUND(
    COUNT(*)::NUMERIC / SUM(COUNT(*)) OVER () * 100,
    2
  ) AS pct_of_total
FROM leavers
GROUP BY exit_type
ORDER BY count DESC;
```
**Kỳ vọng:** Voluntary 70 (70%), Involuntary 30 (30%)

### 4.2 Phòng ban có nhiều người rời nhất
```sql
SELECT 
  d.dept_name,
  COUNT(l.emp_id) AS leavers
FROM leavers l
JOIN employees e 
ON l.emp_id = e.emp_id
JOIN departments d
ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY leavers DESC;
```
**Kỳ vọng:** 

### 4.3 Lý do nghỉ việc phổ biến nhất ?
```sql
SELECT 
  reason,
  COUNT(*) AS count,
  ROUND(
    AVG(
      EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date))
    ),
    1
  ) AS avg_tenure_years
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
GROUP BY reason
ORDER BY count DESC;
```
**Kỳ vọng:** 

**Giải thích:**


---

## 💪 BƯỚC 5: 5 Bài tập đơn giản

### Bài 1: Hiển thị 10 nhân viên rời công ty gần đây nhất.
**Yêu cầu:** Viết query hiển thị 10 nhân viên rời công ty gần đây nhất.

**Đáp án:**
```sql
SELECT 
  emp_id,
  leaving_date,
  reason,
  exit_type
FROM leavers
ORDER BY leaving_date DESC
LIMIT 10;
```


### Bài 2: Số nhân viên rời theo phòng ban
**Yêu cầu:** Viết query đếm số nhân viên rời theo tháng phòng ban.

**Đáp án:**
```sql
SELECT 
  d.dept_name,
  COUNT(l.emp_id) AS leavers
FROM leavers l
JOIN employees e 
ON l.emp_id = e.emp_id
JOIN departments d
ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY leavers DESC;
```

**Giải thích:** 

---

### Bài 3: Tenure của người rời
**Yêu cầu:** Viết query hiển thị nhân viên rời và số năm làm việc..

**Đáp án:**
```sql
SELECT 
  l.emp_id,
  e.hired_date,
  l.leaving_date,
  EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) AS tenure_years
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
ORDER BY tenure_years DESC;
```

**Giải thích:** `CASE WHEN ... IN ... THEN` phân loại lý do rời.

---

### Bài 4: Lương trung bình người rời
**Yêu cầu:** Viết query tính lương trung bình của nhân viên đã rời công ty.

**Đáp án:**
```sql
SELECT 
  ROUND(AVG(s.base_salary),0) AS avg_salary
FROM leavers l
JOIN salaries s
ON l.emp_id = s.emp_id;
```

**Giải thích:** `JOIN salaries` để lấy lương. `AVG(base_salary)` tính trung bình.

---

### Bài 5: Phòng ban có nhiều Involuntary Exit nhất
**Yêu cầu:** Tìm loại exit phổ biến nhất ở mỗi phòng ban.

**Đáp án:**
```sql
SELECT 
  d.dept_name,
  COUNT(*) AS voluntary_leavers
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
JOIN departments d
ON e.dept_id = d.dept_id
WHERE l.exit_type = 'Involuntary'
GROUP BY d.dept_name
ORDER BY voluntary_leavers DESC;
```

**Giải thích:** JOIN 3 bảng, GROUP BY dept_name, đếm leavers.

---

## 🚀 BƯỚC 6: 4 Bài tập nâng cao

### Bài 6: Top 3 phòng ban churn cao nhất
**Yêu cầu:** Viết query tìm ra Top 3 phòng ban churn cao nhất.

**Đáp án:**
```sql
SELECT 
  dept_name,
  leavers,
  total_employees,
  churn_rate_pct
FROM (
  SELECT 
    d.dept_name,
    COUNT(l.emp_id) AS leavers,
    COUNT(e.emp_id) AS total_employees,
    ROUND(
      COUNT(l.emp_id)::NUMERIC /
      COUNT(e.emp_id) * 100,
      2
    ) AS churn_rate_pct
  FROM employees e
  LEFT JOIN leavers l
  ON e.emp_id = l.emp_id
  JOIN departments d
  ON e.dept_id = d.dept_id
  GROUP BY d.dept_name
) t
ORDER BY churn_rate_pct DESC
LIMIT 3;
```

**Giải thích:** 

---

### Bài 7: Xu hướng nghỉ việc theo quý
**Yêu cầu:** Viết query tìm ra Xu hướng nghỉ việc theo quý.

**Đáp án:**
```sql
WITH quarterly_leavers AS (
  SELECT 
    DATE_TRUNC('quarter', leaving_date) AS leave_quarter,
    COUNT(*) AS leavers
  FROM leavers
  GROUP BY leave_quarter
)

SELECT 
  leave_quarter,
  leavers,
  LAG(leavers) OVER (ORDER BY leave_quarter) AS prev_quarter_leavers,
  ROUND(
    (leavers - LAG(leavers) OVER (ORDER BY leave_quarter))::NUMERIC
    / LAG(leavers) OVER (ORDER BY leave_quarter) * 100,
    2
  ) AS qoq_change_pct
FROM quarterly_leavers
ORDER BY leave_quarter;
```

**Giải thích:** 

---

### Bài 8: Risk group - Rời sớm quá
**Yêu cầu:** Viết query tìm nhân viên rời trong vòng < 1 năm (High Risk = may bỏ ngang).

**Đáp án:**
```sql
SELECT 
  COUNT(
    CASE 
      WHEN AGE(l.leaving_date, e.hired_date) < INTERVAL '1 year'
      THEN 1
    END
  ) AS early_leavers,
  COUNT(*) AS total_leavers,
  ROUND(
    COUNT(
      CASE 
        WHEN AGE(l.leaving_date, e.hired_date) < INTERVAL '1 year'
        THEN 1
      END
    )::NUMERIC / COUNT(*) * 100,
    2
  ) AS early_attrition_pct
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id;
```

**Giải thích:** KPI: < 1 năm rời = High Risk (loss / investment cost cao).

---

### Bài 9: Phòng ban có tỷ lệ nghỉ việc cao nhất
**Yêu cầu:** Viết query tính churn rate của phòng ban có tỷ lệ nghỉ việc cao nhất.

**Đáp án:**
```sql
SELECT 
  d.dept_name,
  COUNT(l.emp_id) AS leavers,
  COUNT(e.emp_id) AS total_employees,
  ROUND(
    COUNT(l.emp_id)::NUMERIC /
    COUNT(e.emp_id) * 100,
    2
  ) AS churn_rate_pct
FROM employees e
LEFT JOIN leavers l
ON e.emp_id = l.emp_id
JOIN departments d
ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY churn_rate_pct DESC;
```

**Giải thích:** `GROUP BY d.dept_name`, tính churn rate per phòng, `LIMIT 3` lấy top 3.

---

## 📝 BƯỚC 7: 5 Câu hỏi Quiz

### Quiz 1 (Cơ bản): EXTRACT Year
**Câu hỏi:** Để lấy năm từ leave_date, query nào đúng?
- A) `SELECT YEAR(leave_date) FROM leavers`
- B) `SELECT EXTRACT(YEAR FROM leave_date) FROM leavers` ✅
- C) `SELECT DATE_PART('year', leave_date) FROM leavers`
- D) `SELECT TO_CHAR(leave_date, 'YYYY') FROM leavers`

**Đáp án:** B ✅

**Giải thích:** PostgreSQL dùng `EXTRACT(YEAR FROM ...)`. Câu C (`DATE_PART`) cũng đúng nhưng B là tiêu chuẩn SQL. Câu A (YEAR) là MySQL, Câu D (TO_CHAR) trả về text.

---

### Quiz 2 (Cơ bản): AGE Function
**Câu hỏi:** `AGE(leave_date, hired_date)` tính cái gì?
- A) Số ngày giữa 2 ngày
- B) Khoảng thời gian (interval) giữa 2 ngày ✅
- C) Hiệu số tuyệt đối
- D) Lỗi syntax

**Đáp án:** B ✅

**Giải thích:** `AGE(date1, date2)` trả về interval (có năm, tháng, ngày). Muốn lấy số ngày dùng `date1 - date2`.

---

### Quiz 3 (Trung cấp): CASE WHEN
**Câu hỏi:** Để phân loại Voluntary vs Involuntary dựa trên `reason` column, query nào đúng?
- A) `SELECT CASE reason IN ('Resigned') THEN 'Voluntary' END FROM leavers`
- B) `SELECT CASE WHEN reason IN ('Resigned', 'Better Opportunity') THEN 'Voluntary' ELSE 'Involuntary' END FROM leavers` ✅
- C) `SELECT IF(reason = 'Resigned', 'Voluntary', 'Involuntary') FROM leavers`
- D) `SELECT reason AS 'Voluntary' FROM leavers WHERE reason IN ('Resigned')`

**Đáp án:** B ✅

**Giải thích:** Câu B dùng `CASE WHEN ... THEN ... ELSE ... END` đúng syntax PostgreSQL. Câu A thiếu WHEN. Câu C là MySQL IF. Câu D sai logic (rename column).

---

### Quiz 4 (Trung cấp): Churn Rate Formula
**Câu hỏi:** Churn rate được tính như thế nào?
- A) Số leavers / số employees
- B) (Số leavers / số employees) × 100 ✅
- C) Số leavers × 100
- D) Số leavers - số employees

**Đáp án:** B ✅

**Giải thích:** Churn rate là tỷ lệ (%) nên phải × 100. Ví dụ: 50 rời / 500 NV = 0.1 × 100 = 10%.

---

### Quiz 5 (Nâng cao): CROSS JOIN for Total
**Câu hỏi:** Để tính churn rate (need total_leavers AND total_employees), nên dùng gì?
- A) `INNER JOIN employees` để match từng nhân viên
- B) `CROSS JOIN employees` để lấy tổng toàn bộ ✅
- C) `LEFT JOIN employees` để giữ tất cả leavers
- D) Subquery trong WHERE

**Đáp án:** B ✅

**Giải thích:** `CROSS JOIN` tạo combination của tất cả rows từ 2 bảng (dùng để lấy tổng toàn bộ). `INNER JOIN` sẽ match từng nhân viên (không cần). `LEFT JOIN` cũng match (không cần). Subquery cũng được nhưng `CROSS JOIN` là cách tường minh nhất.

---

## 📖 Tài liệu tham khảo & Links

### PostgreSQL DATE Functions & CASE WHEN
- 🔗 [DATE Functions - PostgreSQL Docs](https://www.postgresql.org/docs/current/functions-datetime.html)
- 🔗 [EXTRACT & AGE Functions - PostgreSQL Manual](https://www.postgresql.org/docs/current/functions-datetime.html#FUNCTIONS-DATETIME-EXTRACT)
- 🔗 [CASE Expression - PostgreSQL Docs](https://www.postgresql.org/docs/current/sql-expressions.html#SQL-CASE)

### Employee Turnover Analytics
- 🔗 [Turnover Rate Calculation - BLS](https://www.bls.gov/opub/hom/pdf/homch5.pdf)
- 🔗 [Voluntary vs Involuntary Separation - SHRM](https://www.shrm.org/)
- 🔗 [Churn Rate Definition & Metrics - HR.com](https://www.hr.com/)

### Cost of Turnover & Retention
- 🔗 [Cost of Employee Turnover - WorkForce Institute](https://www.workforceinstitute.org/)
- 🔗 [Retention Strategies - McKinsey HR](https://www.mckinsey.com/industries/human-capital/)

---

## 🎓 Kết luận Buổi 4

✅ **Bạn vừa hoàn thành:**
- Dùng DATE functions (EXTRACT, AGE) để tính thời gian rời bỏ
- Sử dụng CASE WHEN để phân loại Voluntary vs Involuntary
- Tính churn rate (%) theo thời gian & phòng ban
- Phân tích chi tiết lý do rời bỏ
- Thực hành 5 bài tập đơn giản + 4 nâng cao
- Ôn tập 5 quiz từ cơ bản → nâng cao

📅 **Buổi 5**: Salary Analysis - Phân tích lương
- Window Functions (RANK, ROW_NUMBER, LAG)
- Gender pay gap (công bằng lương)
- Salary history & trends
- Percentile analysis

---

💡 **Ghi chú quan trọng:**
- DATE functions (EXTRACT, AGE) là bắt buộc để phân tích thời gian rời bỏ
- CASE WHEN giúp phân loại logic phức tạp (Voluntary vs Involuntary)
- Churn rate (%) là KPI chính để track health của company
- Turnover cao trong năm `< 1 năm = mất investment tuyển/đào tạo
- Luôn tính churn rate theo phòng để identify problem areas

🚀 **Hãy thực hành các bài tập trên để quen tay với SQL và HR Analytics!**